# Embed rebuilt BoABot chunks on a Colab T4 GPU

This notebook embeds the changed chunks exported by `scripts/rebuild_chunks.py` and downloads a JSONL file that the local rebuild script can reload. In Colab, select **Runtime → Change runtime type → T4 GPU** before running the notebook.

## Input file

Upload the JSONL created with `--export-input`: one chunk per line, with at least `id` and `text` fields. The optional `doc`, `article`, and `status` fields are ignored. No database, credentials, or other local files are needed.

In [ ]:
!pip install -q sentence-transformers

import json
import torch
from sentence_transformers import SentenceTransformer
from google.colab import files

In [ ]:
assert torch.cuda.is_available(), 'No GPU detected. Set Runtime → Change runtime type → T4 GPU, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Choose the chunks_input.jsonl file when prompted.')
input_filename = next(iter(uploaded))
print('Using input:', input_filename)

In [ ]:
with open(input_filename, encoding='utf-8') as input_file:
    rows = [json.loads(line) for line in input_file if line.strip()]

if not rows or any('id' not in row or 'text' not in row for row in rows):
    raise ValueError('Input JSONL must contain one non-empty object per line with id and text.')

model = SentenceTransformer('BAAI/bge-m3', device='cuda')
batch_size = 32
embeddings = []
for start in range(0, len(rows), batch_size):
    batch = rows[start:start + batch_size]
    vectors = model.encode(
        [row['text'] for row in batch],
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    embeddings.extend(vectors)
    print(f'Encoded {min(start + batch_size, len(rows))}/{len(rows)} chunks')

In [ ]:
output_filename = 'chunks_embeddings.jsonl'
with open(output_filename, 'w', encoding='utf-8') as output_file:
    for row, vector in zip(rows, embeddings, strict=True):
        json.dump({'id': row['id'], 'embedding': [float(value) for value in vector]}, output_file)
        output_file.write('\n')

print(f'Wrote {len(rows)} embeddings ({__import__("os").path.getsize(output_filename):,} bytes) to {output_filename}')
files.download(output_filename)

## Run the full pipeline

1. On the local machine, export the changed chunk text without writing to the database:

   `BOABOT_DSN=... .venv/bin/python scripts/rebuild_chunks.py --export-input chunks_input.jsonl --dry-run`

2. Upload `chunks_input.jsonl` here and run all cells. Download the resulting `chunks_embeddings.jsonl`.

3. Back on the local machine, place the downloaded output in the project directory and reload it:

   `.venv/bin/python scripts/rebuild_chunks.py --embeddings-file chunks_embeddings.jsonl`